## 1. Configuration

In [14]:
import json, itertools, gc, time, csv, random
import numpy as np, pandas as pd, lightgbm as lgb

PATH      = "/kaggle/input/competitions/otto-recommender-system/train.jsonl"
TEST_PATH = "/kaggle/input/competitions/otto-recommender-system/test.jsonl"
OUT_PATH  = "/kaggle/working/submission.csv"

N_VAL, N_RANK = 20_000, 150_000      # validation / entraînement LightGBM ; le reste du fichier sert à la co-visitation
COV_START = N_VAL + N_RANK
CHUNK = 50_000
TYPES, TYPE_ID = ["clicks", "carts", "orders"], {"clicks": 0, "carts": 1, "orders": 2}
TYPE_W = np.array([1.0, 3.0, 6.0], dtype=np.float32)
W_OWN = (1.0, 4.0, 4.0)
WINDOW_SEC, MAX_EV, K_INTER = 3600, 30, 40
K_NB, LAST_N, N_NB, MAX_HIST, MAX_OWN = 20, 5, 20, 100, 30
MAX_AID = 2_000_000

## 2. Co-visitation sur les données complètes

In [15]:
def read_sessions(start, stop):
    with open(PATH) as f:
        return [json.loads(l) for l in itertools.islice(f, start, stop)]

def lines_to_df(lines):
    S, A, T, Y = [], [], [], []
    for line in lines:
        s = json.loads(line)
        for e in s["events"]:
            S.append(s["session"]); A.append(e["aid"]); T.append(e["ts"] // 1000); Y.append(TYPE_ID[e["type"]])
    return pd.DataFrame({"session": np.array(S, np.int32), "aid": np.array(A, np.int32),
                         "ts": np.array(T, np.int32), "type": np.array(Y, np.int8)})

def covisit_chunk(df):
    df = df.sort_values(["session", "ts"])
    df["n"] = df.groupby("session").cumcount(ascending=False)
    df = df[df.n < MAX_EV][["session", "aid", "ts", "type"]]
    m = df.merge(df, on="session", suffixes=("_x", "_y"))
    m = m[(m.aid_x != m.aid_y) & ((m.ts_x - m.ts_y).abs() < WINDOW_SEC)]
    m["w"] = TYPE_W[m["type_y"].values]
    return m.groupby(["aid_x", "aid_y"], as_index=False)["w"].sum()

def merge_prune(parts, k):
    df = pd.concat(parts).groupby(["aid_x", "aid_y"], as_index=False)["w"].sum()
    return df.sort_values(["aid_x", "w"], ascending=[True, False]).groupby("aid_x").head(k)

t0, pop, parts = time.time(), np.zeros(MAX_AID, dtype=np.int64), []
with open(PATH) as f:
    for _ in itertools.islice(f, COV_START):
        pass
    i = 0
    while True:
        lines = list(itertools.islice(f, CHUNK))
        if not lines:
            break
        df = lines_to_df(lines)
        pop += np.bincount(df.aid.values, minlength=MAX_AID)
        parts.append(covisit_chunk(df))
        if len(parts) >= 4:
            parts = [merge_prune(parts, K_INTER)]
        gc.collect(); i += 1
        if i % 20 == 0:
            print(f"{i * CHUNK:>10,} sessions | {(time.time() - t0) / 60:.1f} min")
cov_df = merge_prune(parts, K_NB)
del parts; gc.collect()

ay, wv, ax = cov_df.aid_y.values, cov_df.w.values, cov_df.aid_x.values
u, st = np.unique(ax, return_index=True)
cov_idx = dict(zip(u.tolist(), zip(st.tolist(), np.append(st[1:], len(ax)).tolist())))
top20 = np.argsort(-pop)[:20].tolist()
print("Co-visitation prête :", len(cov_idx), "produits |", round((time.time() - t0) / 60, 1), "min")

 1,000,000 sessions | 3.1 min
 2,000,000 sessions | 7.1 min
 3,000,000 sessions | 10.9 min
 4,000,000 sessions | 15.3 min
 5,000,000 sessions | 19.9 min
 6,000,000 sessions | 24.2 min
 7,000,000 sessions | 29.3 min
 8,000,000 sessions | 34.5 min
 9,000,000 sessions | 39.2 min
10,000,000 sessions | 44.7 min
11,000,000 sessions | 50.2 min
12,000,000 sessions | 55.0 min
Co-visitation prête : 1828028 produits | 59.7 min


## 3. Candidats, features et classement

In [21]:
FEATS = ["n_click", "n_cart", "n_order", "last_r", "first_r", "own_score", "mins_last",
         "nb_score", "nb_cnt", "nb_rank", "log_pop", "is_own", "n_events", "n_unique",
         "last_type", "sess_min"]

def split_session(events):
    """Coupure pseudo-aléatoire (validation locale)."""
    if len(events) < 2:
        return events, []
    cut = random.Random(len(events) * 7919 + events[0]["ts"] % 1000).randint(1, len(events) - 1)
    return events[:cut], events[cut:]

def session_features(history):
    history = history[-MAX_HIST:]
    t_end = history[-1]["ts"] / 1000
    own = {}
    for r, e in enumerate(reversed(history)):
        a, ti = e["aid"], TYPE_ID[e["type"]]
        d = own.get(a)
        if d is None:
            d = own[a] = [0, 0, 0, r, r, 0.0, (t_end - e["ts"] / 1000) / 60]
        d[ti] += 1; d[4] = r; d[5] += W_OWN[ti] / (1 + r)
    nb_score, nb_cnt = {}, {}
    for r, e in enumerate(reversed(history[-LAST_N:])):
        se = cov_idx.get(e["aid"])
        if se is None:
            continue
        ids, ws = ay[se[0]:se[1]], wv[se[0]:se[1]]
        for a, w in zip(ids.tolist(), (ws / ws[0]).tolist()):
            nb_score[a] = nb_score.get(a, 0.0) + w / (1 + r)
            nb_cnt[a] = nb_cnt.get(a, 0) + 1
    nb_sorted = sorted(nb_score, key=nb_score.get, reverse=True)
    nb_rank = {a: i for i, a in enumerate(nb_sorted)}
    own_top = sorted(own, key=lambda a: own[a][5], reverse=True)[:MAX_OWN]
    own_set = set(own_top)
    cands = own_top + [a for a in nb_sorted if a not in own_set][:N_NB]
    n, n_unique = len(history), len(own)
    last_type = TYPE_ID[history[-1]["type"]]
    sess_min = (t_end - history[0]["ts"] / 1000) / 60
    X = np.empty((len(cands), len(FEATS)), dtype=np.float32)
    for i, a in enumerate(cands):
        d = own.get(a)
        if d:
            c, ca, o, last_r, first_r, sc, ml = d
        else:
            c = ca = o = 0; last_r = first_r = 999; sc = 0.0; ml = 9999
        X[i] = (c, ca, o, last_r, first_r, sc, ml, nb_score.get(a, 0.0), nb_cnt.get(a, 0),
                nb_rank.get(a, 999), np.log1p(pop[a]) if a < MAX_AID else 0.0, d is not None,
                n, n_unique, last_type, sess_min)
    return cands, X

def predict_batch(histories, scorers):
    cand, feats = [], []
    for h in histories:
        a, X = session_features(h); cand.append(a); feats.append(X)
    Xall, bounds = np.vstack(feats), np.cumsum([0] + [len(a) for a in cand])
    out = [{} for _ in cand]
    for t in TYPES:
        p = scorers[t](Xall)
        for i, a in enumerate(cand):
            lst = [a[j] for j in np.argsort(-p[bounds[i]:bounds[i + 1]])[:20]]
            lst += [x for x in top20 if x not in lst][:20 - len(lst)]
            out[i][t] = lst
    return out

## 4. Entraînement LightGBM et score de validation locale

In [22]:
Xs, ys = [], {t: [] for t in TYPES}
for s in read_sessions(N_VAL, N_VAL + N_RANK):
    hist, fut = split_session(s["events"])
    if not fut:
        continue
    a, X = session_features(hist); Xs.append(X)
    for t in TYPES:
        truth = [e["aid"] for e in fut if e["type"] == t]
        truth = set(truth[:1]) if t == "clicks" else set(truth)
        ys[t].append(np.array([x in truth for x in a], dtype=np.int8))
X_tr = np.vstack(Xs)
models = {t: lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, subsample=0.8,
                                subsample_freq=1, colsample_bytree=0.8, random_state=0, verbose=-1
                                ).fit(X_tr, np.concatenate(ys[t])) for t in TYPES}
scorers = {t: (lambda X, m=models[t]: m.predict_proba(X)[:, 1]) for t in TYPES}

# Validation locale : mêmes formules que Kaggle (somme globale des hits / somme de min(20, nb vrais))
pairs = [split_session(s["events"]) for s in read_sessions(0, N_VAL)]
pairs = [(h, f) for h, f in pairs if f]
preds = predict_batch([h for h, _ in pairs], scorers)
hits, den = dict.fromkeys(TYPES, 0), dict.fromkeys(TYPES, 0)
for p, (_, fut) in zip(preds, pairs):
    for t in TYPES:
        truth = [e["aid"] for e in fut if e["type"] == t]
        truth = set(truth[:1]) if t == "clicks" else set(truth)
        hits[t] += len(set(p[t]) & truth); den[t] += min(20, len(truth))
R = {t: hits[t] / den[t] for t in TYPES}
print({t: round(v, 4) for t, v in R.items()}, "Score :", round(0.1 * R["clicks"] + 0.3 * R["carts"] + 0.6 * R["orders"], 4))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'clicks': 0.426, 'carts': 0.16, 'orders': 0.3202} Score : 0.2827


## 5. Soumission

In [23]:
t0, n_done = time.time(), 0
with open(TEST_PATH) as f, open(OUT_PATH, "w", newline="") as out:
    w = csv.writer(out)
    w.writerow(["session_type", "labels"])
    while True:
        lines = list(itertools.islice(f, 20_000))
        if not lines:
            break
        sess = [json.loads(l) for l in lines]
        for s, p in zip(sess, predict_batch([s["events"] for s in sess], scorers)):
            for t in TYPES:
                w.writerow([f"{s['session']}_{t}", " ".join(map(str, p[t]))])
        n_done += len(sess)
        print(f"{n_done:,} sessions | {(time.time() - t0) / 60:.1f} min")
print("submission.csv écrit")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


20,000 sessions | 0.2 min


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


40,000 sessions | 0.5 min


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


60,000 sessions | 0.7 min


... (158 outputs left out)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1,660,000 sessions | 19.6 min


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1,671,803 sessions | 19.7 min
submission.csv écrit
